# 11장. 로지스틱 회귀

예측이 항상 0과 1 사이에 있는 모델은 어떻게 만드는가.

**준비:** 1장에서 쓴 한국전력거래소 「시간별 전국 전력수요량」 CSV 세 개와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개, 6장에서 쓴 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 3분쯤 걸린다.

## 11.1 선형 회귀로 확률을 예측할 때의 문제

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()

LINE = 74                                            # 일평균 수요가 74GW 이상이면 '수요가 많은 날'로 정한다
y23, y25 = (Y23 >= LINE).astype(float), (Y25 >= LINE).astype(float)
print(len(y23), int(y23.sum()), round(float(y23.mean()), 3), len(y25), int(y25.sum()), round(float(y25.mean()), 3))

In [ ]:
def bends(t):
    """12도보다 낮은 만큼과 20도보다 높은 만큼. 1장의 꺾은 직선에서 쓴 두 특성이다."""
    return np.column_stack([np.maximum(12 - t, 0), np.maximum(t - 20, 0)])

def with_one(M):
    return np.column_stack([np.ones(len(M)), M])

straight = np.linalg.lstsq(with_one(bends(T23)), y23, rcond=None)[0]
said23, said25 = with_one(bends(T23)) @ straight, with_one(bends(T25)) @ straight
print(straight.round(3).tolist())
print(round(float(said23.min()), 3), round(float(said23.max()), 3), int((said23 < 0).sum()), int((said23 > 1).sum()))
print(round(float(said25.min()), 3), round(float(said25.max()), 3), int((said25 < 0).sum()), int((said25 > 1).sum()))
mild = (T23 > 12) & (T23 < 20)
print(int(mild.sum()), int(y23[mild].sum()), round(float(said23[mild].mean()), 3))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 11-1
xs = np.linspace(-9, 31, 400)
jitter = np.random.default_rng(0).uniform(-0.03, 0.03, len(y23))
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.axhspan(-0.2, 0, color=to_rgba(BRICK, 0.10), lw=0)
ax.scatter(T23, y23 + jitter, s=11, color=to_rgba(NAVY, 0.45), edgecolor="none")
ax.plot(xs, with_one(bends(xs)) @ straight, color=BRICK, lw=1.9)
ax.axhline(0, color=GRAY, lw=0.8); ax.axhline(1, color=GRAY, lw=0.8)
ax.text(-8.6, 1.07, "74GW 이상인 날(1)", fontsize=8, color=NAVY, va="bottom")
ax.text(16, 0.07, "74GW 미만인 날(0)", fontsize=8, color=NAVY, va="bottom", ha="center")
ax.annotate(f"12~20도인 {int(mild.sum())}일의 예측은 모두 {straight[0]:.3f}", (16, straight[0]), (16, -0.165), fontsize=8, color=BRICK, ha="center", arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.text(2.2, 0.52, "선형 회귀의 예측", fontsize=8, color=BRICK)
ax.set_xlim(-9.5, 31.5); ax.set_ylim(-0.2, 1.18)
ax.set_xlabel("일평균 기온(°C)", fontsize=8)
ax.set_ylabel("수요가 많은 날일 확률", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_1.png", dpi=300)
plt.show()

In [ ]:
def sigmoid(z):
    """어떤 실수든 0과 1 사이의 값으로 바꾼다. 0이면 0.5, 클수록 1에, 작을수록 0에 가까워진다."""
    return 1 / (1 + np.exp(-z))

probs = np.array([0.01, 0.1, 0.5, 0.9, 0.99])
odds = probs / (1 - probs)
print(pd.DataFrame({"확률": probs, "승산": odds.round(3), "로그 승산": np.log(odds).round(2),
                    "시그모이드(로그 승산)": sigmoid(np.log(odds)).round(2)}).to_string(index=False))

In [ ]:
# 그림 11-2
z = np.linspace(-7, 7, 400)
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(z, sigmoid(z), color=NAVY, lw=2)
ax.axhline(0, color=GRAY, lw=0.8); ax.axhline(1, color=GRAY, lw=0.8)
for pr, tx, ty, ha in [(0.01, -6.8, 0.09, "left"), (0.1, -1.8, 0.07, "left"), (0.5, 0.45, 0.47, "left"), (0.9, 2.6, 0.84, "left"), (0.99, 4.3, 1.05, "right")]:
    zz = np.log(pr / (1 - pr))
    ax.scatter([zz], [pr], s=36, color=BRICK, zorder=5)
    ax.text(tx, ty, f"로그 승산 {zz:.1f} → 확률 {pr:g}", fontsize=8, color=BRICK, ha=ha, va="center")
ax.set_xlim(-7, 7); ax.set_ylim(-0.06, 1.12)
ax.set_xlabel("z (로그 승산)", fontsize=8)
ax.set_ylabel("시그모이드(z) = 확률", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_11_2.png", dpi=300)
plt.show()

## 11.2 로그 손실과 경사하강법: 계수의 추정과 해석

In [ ]:
def log_loss(actual, prob):
    """실제로 일어난 결과에 예측한 확률의 로그를 구해 부호를 바꾸고 평균한다. 0에 가까울수록 좋다."""
    prob = np.clip(prob, 1e-12, 1 - 1e-12)
    return float(-(actual * np.log(prob) + (1 - actual) * np.log(1 - prob)).mean())

said = np.array([0.99, 0.9, 0.5, 0.1, 0.01, 0.001])
print(pd.DataFrame({"일어난 결과에 예측한 확률": said, "제곱 오차": ((1 - said) ** 2).round(3),
                    "로그 손실": (-np.log(said)).round(3)}).to_string(index=False))
print(round(log_loss(y23, np.full(len(y23), y23.mean())), 4))

In [ ]:
# 그림 11-4
pr = np.linspace(0.001, 1, 500)
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(pr, -np.log(pr), color=BRICK, lw=1.9)
ax.plot(pr, (1 - pr) ** 2, color=NAVY, lw=1.9)
ax.text(0.2, 5.6, "로그 손실: 확률이 0에 가까울수록\n한없이 커진다", fontsize=8, color=BRICK, linespacing=1.5)
ax.text(0.02, 0.25, "제곱 오차: 1을 넘지 않는다", fontsize=8, color=NAVY)
for q in [0.1, 0.01]:
    ax.scatter([q], [-np.log(q)], s=30, color=BRICK, zorder=5)
ax.text(0.115, 2.38, "0.1이면 2.30", fontsize=8, color=BRICK)
ax.text(0.03, 4.6, "0.01이면 4.61", fontsize=8, color=BRICK)
ax.set_xlim(0, 1); ax.set_ylim(0, 7)
ax.set_xlabel("실제로 일어난 결과에 예측한 확률", fontsize=8)
ax.set_ylabel("그 한 건의 손실", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_4.png", dpi=300)
plt.show()

In [ ]:
B = bends(T23)
mu, sd = B.mean(axis=0), B.std(axis=0)
Z, Z25 = with_one((B - mu) / sd), with_one((bends(T25) - mu) / sd)       # 10장에서 본 대로 표준화한다

def descend(M, actual, rate, steps):
    """계수 0에서 시작해서 로그 손실의 기울기 반대 방향으로 이동한다."""
    coef, seen = np.zeros(M.shape[1]), []
    for _ in range(steps):
        slope = M.T @ (sigmoid(M @ coef) - actual) / len(actual)         # 로그 손실을 계수마다 미분한 값
        coef = coef - rate * slope
        seen.append(log_loss(actual, sigmoid(M @ coef)))
    return coef, seen

tracks = {}
for rate in [0.1, 1, 5]:
    coef, tracks[rate] = descend(Z, y23, rate, 5000)
    print(rate, [round(tracks[rate][i], 4) for i in (0, 9, 99, 999, 4999)], coef.round(3).tolist())
curve = np.linalg.eigvalsh(Z.T @ Z / len(y23))
print(curve.round(3).tolist(), round(float(8 / curve[-1]), 2))

In [ ]:
# 그림 11-5
fig, ax = plt.subplots(figsize=(7, 3.9))
steps = np.arange(1, 5001)
for rate, col in [(0.1, GRAY), (1, NAVY), (5, BRICK)]:
    ax.plot(steps, tracks[rate], color=col, lw=1.8)
ax.axhline(tracks[5][-1], color=TEAL, lw=1.1, ls=(0, (4, 2)))
ax.text(1.1, tracks[5][-1] - 0.012, f"최솟값 {tracks[5][-1]:.4f}", fontsize=8, color=TEAL, va="top")
ax.text(60, 0.47, "학습률 0.1", fontsize=8, color="#5b6570")
ax.text(9, 0.36, "학습률 1", fontsize=8, color=NAVY)
ax.text(1.15, 0.232, "학습률 5", fontsize=8, color=BRICK)
ax.set_xscale("log"); ax.set_xlim(0.9, 5500); ax.set_ylim(0.15, 0.72)
plain_ticks(ax.xaxis)
ax.set_xlabel("이동 횟수(로그 축)", fontsize=8)
ax.set_ylabel("로그 손실", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_5.png", dpi=300)
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

theirs = LogisticRegression(C=1e8, max_iter=10000).fit(Z[:, 1:], y23)      # C가 아주 크면 규제가 거의 없다
print(theirs.intercept_.round(3).tolist(), theirs.coef_.round(3).tolist(), round(log_loss(y23, theirs.predict_proba(Z[:, 1:])[:, 1]), 4))

fitted = sigmoid(Z @ coef)
print((Z.T @ (fitted - y23)).round(4).tolist())
print(round(float(fitted.sum()), 2), int(y23.sum()))

In [ ]:
per_degree = coef[1:] / sd                              # 표준화한 계수를 1도당 계수로 되돌린다
print(per_degree.round(3).tolist(), np.exp(per_degree).round(2).tolist())

def prob_at(t):
    return sigmoid(with_one((bends(np.atleast_1d(t).astype(float)) - mu) / sd) @ coef)

look = np.array([-5.0, 0, 5, 10, 15, 20, 24, 26, 28, 30])
print(pd.DataFrame({"기온": look, "로지스틱": prob_at(look).round(3), "선형 회귀": (with_one(bends(look)) @ straight).round(3)}).T.to_string(header=False))

rows = []
for t in [22, 24, 26, 28, 30, 32]:
    p0, p1 = float(prob_at(t)[0]), float(prob_at(t + 1)[0])
    rows.append((f"{t}→{t + 1}도", round(p0, 3), round(p1, 3), round((p1 - p0) * 100, 1), round(p1 / (1 - p1) / (p0 / (1 - p0)), 2)))
print(pd.DataFrame(rows, columns=["기온 변화", "앞 확률", "뒤 확률", "확률 변화(%p)", "승산의 배수"]).to_string(index=False))

In [ ]:
# 그림 11-6
fig, ax = plt.subplots(figsize=(7, 4.1))
ax.scatter(T23, y23 + jitter, s=11, color=to_rgba(GRAY, 0.6), edgecolor="none")
ax.plot(xs, prob_at(xs), color=NAVY, lw=2, label="로지스틱 회귀")
ax.plot(xs, with_one(bends(xs)) @ straight, color=BRICK, lw=1.6, ls=(0, (4, 2)), label="선형 회귀")
ax.legend(frameon=False, fontsize=8, loc="center", bbox_to_anchor=(0.42, 0.62))
ax.axhline(0, color=GRAY, lw=0.8); ax.axhline(1, color=GRAY, lw=0.8)
for t, off in [(24, (-6.5, 0.2)), (28, (-7.5, 0.16))]:
    p0, p1 = float(prob_at(t)[0]), float(prob_at(t + 1)[0])
    ax.plot([t, t + 1, t + 1], [p0, p0, p1], color=TEAL, lw=1.4)
    ax.annotate(f"{t}→{t + 1}도: +{(p1 - p0) * 100:.1f}%p", (t + 1, (p0 + p1) / 2), (t + off[0], (p0 + p1) / 2 + off[1]), fontsize=8, color=TEAL, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.set_xlim(-9.5, 31.5); ax.set_ylim(-0.2, 1.12)
ax.set_xlabel("일평균 기온(°C)", fontsize=8)
ax.set_ylabel("수요가 많은 날일 확률", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_6.png", dpi=300)
plt.show()

In [ ]:
def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

def brier(actual, prob):
    return float(((prob - actual) ** 2).mean())

p25 = sigmoid(Z25 @ coef)
base = np.full(len(y25), y23.mean())
print("기준율    ", round(brier(y25, base), 4), round(log_loss(y25, base), 4))
print("선형 회귀 ", round(auc(y25, said25), 3), round(brier(y25, said25), 4))
print("로지스틱  ", round(auc(y25, p25), 3), round(brier(y25, p25), 4), round(log_loss(y25, p25), 4))
bins = pd.cut(p25, [0, 0.05, 0.2, 0.5, 0.8, 1])
shelf25 = pd.DataFrame({"예측": p25, "실제": y25}).groupby(bins, observed=True).agg(날=("실제", "size"), 예측_평균=("예측", "mean"), 실제_비율=("실제", "mean")).round(3)
print(shelf25.to_string())

In [ ]:
# 그림 11-7
fig, ax = plt.subplots(figsize=(7, 4.1))
ax.plot([0, 1], [0, 1], color=GRAY, lw=1, ls=(0, (4, 2)))
ax.plot(shelf25["예측_평균"], shelf25["실제_비율"], color=NAVY, lw=1.6, marker="o", ms=6)
places = [(0.03, -0.055, "left"), (0.035, -0.045, "left"), (-0.03, 0.05, "right"), (0.035, -0.04, "left"), (0.03, -0.05, "left")]
for (pm, ar, cnt), (dx, dy, ha) in zip(shelf25[["예측_평균", "실제_비율", "날"]].to_numpy(), places):
    ax.text(pm + dx, ar + dy, f"{int(cnt)}일", fontsize=8, color=NAVY, ha=ha, va="center")
ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.08, 1.03)
ax.set_xlabel("예측 확률의 구간별 평균", fontsize=8)
ax.set_ylabel("2025년에 실제로 74GW 이상이었던 날의 비율", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_11_7.png", dpi=300)
plt.show()

## 11.3 scikit-learn의 기본 규제와 완전 분리

In [ ]:
plain = LogisticRegression().fit(Z[:, 1:], y23)                       # 설정을 하나도 주지 않았다
print(plain.intercept_.round(3).tolist(), plain.coef_.round(3).tolist(), plain.C)

raw_default = LogisticRegression(max_iter=10000).fit(B, y23)          # 표준화하지 않은 특성에 기본 설정
raw_free = LogisticRegression(C=1e8, max_iter=10000).fit(B, y23)
print(raw_default.coef_.round(3).tolist(), raw_free.coef_.round(3).tolist())
print((plain.coef_[0] / sd).round(3).tolist())

In [ ]:
name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
def town(df):
    """지번 주소에서 시군(구)과 읍면동까지만 잘라 낸다."""
    addr = df["소재지지번주소"].fillna("").str.replace(r"\s+", " ", regex=True)
    return addr.str.extract(r"^경기도 (\S+(?: \S+구)? \S+?[읍면동가리])(?:\s|$)")[0].fillna("?")

year = train["기준일"].dt.year.to_numpy()
fit_on, tune_on = year < 2020, year == 2020              # 계수를 추정하는 줄, 규제 세기를 고르는 줄
TOWNS = sorted(town(train)[fit_on].unique())
W, Wt = np.column_stack([X, onehot(town(train), TOWNS)]), np.column_stack([Xt, onehot(town(test), TOWNS)])

tally = pd.DataFrame({"읍면동": town(train)[fit_on].to_numpy(), "폐업함": ytr[fit_on]}).groupby("읍면동")["폐업함"].agg(["size", "sum"])
never = tally.index[tally["sum"] == 0]                   # 추정에 쓰는 줄에서 폐업이 한 건도 없는 읍면동
print(W.shape, len(never), int(tally.loc[never, "size"].sum()), int((tally["sum"] == tally["size"]).sum()))
there = town(train)[tune_on].isin(never).to_numpy()
print(int(there.sum()), int(ytr[tune_on][there].sum()), round(float(ytr[tune_on][there].mean()), 3))

In [ ]:
never_cols = 41 + np.where(np.isin(TOWNS, never))[0]
rows, frees = [], {}
for tol in [1e-4, 1e-6, 1e-8]:
    # C를 아주 크게 해서 규제를 없애고, 계산을 멈추는 기준(tol)만 점점 엄격하게 한다
    m = LogisticRegression(C=1e8, max_iter=100000, tol=tol).fit(W[fit_on], ytr[fit_on])
    frees[tol] = m
    said = m.predict_proba(W[tune_on])[:, 1]
    rows.append((f"{tol:.0e}", int(m.n_iter_[0]), round(float(m.coef_[0][never_cols].mean()), 2), round(float(m.coef_[0][never_cols].min()), 2),
                 f"{said[there].mean():.1e}", round(log_loss(ytr[fit_on], m.predict_proba(W[fit_on])[:, 1]), 4), round(log_loss(ytr[tune_on], said), 4)))
print(pd.DataFrame(rows, columns=["tol", "반복 횟수", "계수 평균", "가장 작은 계수", "예측 확률", "훈련 손실", "2020년 손실"]).set_index("tol").to_string())
print(round(log_loss(ytr[tune_on], np.full(tune_on.sum(), ytr[fit_on].mean())), 4))

In [ ]:
# 그림 11-8
m0 = frees[1e-4]
rows_never = np.isin(town(train)[fit_on].to_numpy(), never)                     # 폐업이 없는 읍면동의 훈련 줄
own = W[fit_on][rows_never][:, never_cols] @ m0.coef_[0][never_cols]            # 줄마다 자기 읍면동의 계수
rest = m0.decision_function(W[fit_on][rows_never]) - own                        # 읍면동 계수를 뺀 나머지
shift = np.linspace(-16, 0, 200)
part = [np.log1p(np.exp(rest + c)).mean() for c in shift]                       # 읍면동 계수가 모두 c일 때 이 줄들의 로그 손실
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(shift, part, color=NAVY, lw=1.9)
marks = [(1e-4, "tol = 0.0001", (-5.6, 0.07)), (1e-6, "tol = 0.000001", (-9.6, 0.045)), (1e-8, "tol = 0.00000001", (-15.4, 0.045))]
for tol, label, where in marks:
    c = float(frees[tol].coef_[0][never_cols].mean())
    val = float(np.log1p(np.exp(rest + c)).mean())
    ax.scatter([c], [val], s=40, color=BRICK, zorder=5)
    ax.annotate(f"{label}\n계수 평균 {c:.1f}", (c, val), where, fontsize=8, color=BRICK, linespacing=1.5, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.text(-15.6, 0.105, f"폐업이 없는 읍면동 {len(never)}곳의 훈련 자료 {int(rows_never.sum())}줄", fontsize=8, color=NAVY)
ax.set_xlim(-16.5, 0.5); ax.set_ylim(-0.008, 0.12)
ax.set_xlabel("폐업이 없는 읍면동의 계수(모두 같은 값으로 두었을 때)", fontsize=8)
ax.set_ylabel("그 줄들의 로그 손실", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_8.png", dpi=300)
plt.show()

In [ ]:
CS = [10, 1, 0.3, 0.1, 0.03, 0.01, 0.003]
rows, logits = [], {}
for C_ in CS:
    logits[C_] = LogisticRegression(C=C_, max_iter=20000).fit(W[fit_on], ytr[fit_on])
    said = logits[C_].predict_proba(W[tune_on])[:, 1]
    rows.append((C_, round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4), round(log_loss(ytr[tune_on], said), 4),
                 round(float(said.min()), 3), round(float(said.max()), 3), round(float(np.abs(logits[C_].coef_[0][never_cols]).mean()), 2)))
picks = pd.DataFrame(rows, columns=["C", "AUC", "브라이어", "로그 손실", "가장 낮은 확률", "가장 높은 확률", "읍면동 계수"]).set_index("C")
print(picks.to_string())

In [ ]:
# 그림 11-9
fig, ax = plt.subplots(figsize=(7, 3.9))
flat = log_loss(ytr[tune_on], np.full(tune_on.sum(), ytr[fit_on].mean()))
ax.plot(picks.index, picks["로그 손실"], color=NAVY, lw=1.8, marker="o", ms=5)
ax.axhline(flat, color=GRAY, lw=1.1, ls=(0, (4, 2)))
ax.text(10, flat + 0.0006, f"기준율로만 예측 {flat:.4f}", fontsize=8, color="#5b6570", ha="right", va="bottom")
best_C = float(picks["로그 손실"].idxmin())
ax.annotate(f"C = {best_C:g}에서 {picks['로그 손실'].min():.4f}", (best_C, picks["로그 손실"].min()), (0.05, 0.4835), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.text(0.0031, 0.4662, "← 규제가 강하다", fontsize=8, color="#5b6570", ha="left")
ax.text(10, 0.4662, "규제가 약하다 →", fontsize=8, color="#5b6570", ha="right")
ax.set_xscale("log"); ax.set_ylim(0.465, 0.4925)
plain_ticks(ax.xaxis)
ax.set_xlabel("C (규제 세기의 역수, 로그 축)", fontsize=8)
ax.set_ylabel("2020년 검증 자료의 로그 손실", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_9.png", dpi=300)
plt.show()

## 11.4 선형 회귀와의 비교, 그리고 class_weight

In [ ]:
line41 = np.linalg.lstsq(X[fit_on], ytr[fit_on], rcond=None)[0]
logit41 = LogisticRegression(C=1e6, max_iter=20000).fit(X[fit_on], ytr[fit_on])
a, b = X[tune_on] @ line41, logit41.predict_proba(X[tune_on])[:, 1]
for name, said in [("선형 회귀", a), ("로지스틱", b)]:
    print(name, round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4), round(log_loss(ytr[tune_on], said), 4),
          round(float(said.min()), 3), round(float(said.max()), 3))
print(round(float(np.corrcoef(a, b)[0, 1]), 4), round(float(np.abs(a - b).mean()), 4), round(float(np.abs(a - b).max()), 3))

In [ ]:
# 그림 11-10
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot([0.05, 0.65], [0.05, 0.65], color=GRAY, lw=1, ls=(0, (4, 2)))
ax.scatter(a, b, s=6, color=to_rgba(NAVY, 0.35), edgecolor="none")
ax.text(0.085, 0.57, f"가게 {len(a):,}곳, 상관계수 {np.corrcoef(a, b)[0, 1]:.4f}\n차이의 평균 {np.abs(a - b).mean():.4f}, 최대 {np.abs(a - b).max():.3f}", fontsize=8, color=NAVY, linespacing=1.6)
ax.set_xlim(0.07, 0.63); ax.set_ylim(0.07, 0.63)
ax.set_xlabel("선형 회귀의 예측", fontsize=8)
ax.set_ylabel("로지스틱 회귀의 예측 확률", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_11_10.png", dpi=300)
plt.show()

In [ ]:
fit41 = logit41.predict_proba(X[fit_on])[:, 1]
print(round(float(fit41.sum()), 1), int(ytr[fit_on].sum()))
for name in ["고양시", "화성시", "수원시"]:
    there_ = (train["시군"].to_numpy()[fit_on] == name)
    print(name, int(there_.sum()), round(float(fit41[there_].sum()), 1), int(ytr[fit_on][there_].sum()))
bins = pd.cut(b, [0, 0.15, 0.2, 0.25, 0.3, 1])
print(pd.DataFrame({"예측": b, "실제": ytr[tune_on]}).groupby(bins, observed=True).agg(가게=("실제", "size"), 예측_평균=("예측", "mean"), 실제_비율=("실제", "mean")).round(3).to_string())

In [ ]:
even = LogisticRegression(C=1e6, max_iter=20000, class_weight="balanced").fit(X[fit_on], ytr[fit_on])
c = even.predict_proba(X[tune_on])[:, 1]
print(round(auc(ytr[tune_on], c), 3), round(brier(ytr[tune_on], c), 4), round(log_loss(ytr[tune_on], c), 4))
print(round(float(b.mean()), 3), round(float(c.mean()), 3), round(float(ytr[tune_on].mean()), 3))
print(int((b >= 0.5).sum()), int((c >= 0.5).sum()), round(float(ytr[tune_on][c >= 0.5].mean()), 3))
pi = ytr[fit_on].mean()
shifted = sigmoid(np.log(c / (1 - c)) + np.log(pi / (1 - pi)))          # 로그 승산에 훈련 자료의 로그 승산을 더한다
print(round(float(shifted.mean()), 3), round(brier(ytr[tune_on], shifted), 4), round(float(np.abs(shifted - b).max()), 3))

In [ ]:
# 그림 11-11
fig, ax = plt.subplots(figsize=(7, 3.9))
edges = np.linspace(0, 0.9, 46)
ax.hist(b, bins=edges, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.8)
ax.hist(c, bins=edges, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.8)
ax.axvline(ytr[tune_on].mean(), color=TEAL, lw=1.3, ls=(0, (4, 2)))
top = ax.get_ylim()[1]
ax.text(ytr[tune_on].mean() + 0.012, top * 0.93, f"실제 폐업 비율 {ytr[tune_on].mean():.3f}", fontsize=8, color=TEAL)
ax.text(0.25, top * 0.62, f"가중치 없음\n평균 {b.mean():.3f}", fontsize=8, color=NAVY, linespacing=1.5)
ax.text(0.62, top * 0.3, f'class_weight="balanced"\n평균 {c.mean():.3f}', fontsize=8, color=BRICK, linespacing=1.5, ha="left")
ax.set_xlim(0, 0.9)
ax.set_xlabel("2020년 검증 자료 가게의 예측 확률", fontsize=8)
ax.set_ylabel("가게 수", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_11_11.png", dpi=300)
plt.show()

In [ ]:
def exam(name, said):
    print(f"{name}: AUC {auc(y, said):.3f} 브라이어 {brier(y, said):.4f} 로그 손실 {log_loss(y, said):.4f}")

flat_t = np.full(len(y), ytr[fit_on].mean())
print(f"기준율로만 예측: 브라이어 {brier(y, flat_t):.4f} 로그 손실 {log_loss(y, flat_t):.4f}")
t_line, t_logit = Xt @ line41, logit41.predict_proba(Xt)[:, 1]
exam("선형 회귀, 특성 41개", t_line)
exam("로지스틱, 특성 41개", t_logit)
exam("로지스틱, 특성 679개, 규제 없음(C=1000000)", LogisticRegression(C=1e6, max_iter=20000).fit(W[fit_on], ytr[fit_on]).predict_proba(Wt)[:, 1])
t_reg = logits[0.03].predict_proba(Wt)[:, 1]
exam("로지스틱, 특성 679개, C=0.03", t_reg)

rng = np.random.default_rng(0)
gaps = []
for _ in range(2000):
    pick = rng.integers(0, len(y), len(y))               # 8.2절의 짝지은 부트스트랩
    gaps.append(auc(y[pick], t_logit[pick]) - auc(y[pick], t_line[pick]))
print(f"로지스틱 - 선형 회귀(특성 41개): AUC 차이 {np.mean(gaps):+.4f} 95% 구간 [{np.quantile(gaps, 0.025) + 0:+.4f}, {np.quantile(gaps, 0.975) + 0:+.4f}]")